# Notebook 3: Production Best Practices

Your agent is deployed — now let's harden it for production. This notebook covers:

1. **Agent configuration** — explicit model settings, conversation management
2. **Security** — least-privilege IAM, input validation, output sanitization
3. **Performance** — streaming, conversation window tuning, error handling
4. **Observability** — logging, metrics, tracing
5. **Cost optimization** — model selection, token management

> **Prerequisites:** Familiarity with [Notebook 01](./01_local_agent_to_http_service.ipynb) and [Notebook 02](./02_deploy_to_aws.ipynb).

In [ ]:
!pip install strands-agents strands-agents-tools

## 1. Production Agent Configuration

In development, defaults are fine. In production, be explicit about every setting.

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands.agent.conversation_manager import SlidingWindowConversationManager
from strands_tools import http_request

# Explicit model configuration — don't rely on defaults in production
model = BedrockModel(
    model_id="anthropic.claude-sonnet-4-20250514-v1:0",
    region_name="us-east-1",
    temperature=0.3,       # Low temperature for factual, consistent responses
    max_tokens=2048,       # Enough room for tool calls + response
    top_p=0.8,            # Slightly constrained sampling
)

# Conversation management prevents context window overflow
conversation_manager = SlidingWindowConversationManager(
    window_size=10,  # Keep last 10 turns; older messages are trimmed
)

# Explicitly list tools — never use auto-discovery in production
agent = Agent(
    model=model,
    tools=[http_request],
    system_prompt="You are a weather assistant. Use the http_request tool to fetch forecasts.",
    conversation_manager=conversation_manager,
)

print("Agent configured for production.")

### Why These Settings Matter

| Setting | Default | Production | Why |
|---------|---------|------------|-----|
| `temperature` | 1.0 | 0.1–0.3 | Consistent, reproducible responses |
| `max_tokens` | Model default | 2048+ | Prevents truncated responses; agents need room for tool calls |
| `window_size` | Unlimited | 10–20 | Prevents context overflow; controls memory costs |
| `tools` | `[]` | Explicit list | Prevents accidental tool exposure; audit trail |

## 2. Security Best Practices

### 2.1 Least-Privilege IAM

Never use `resources: ['*']` in production. Scope permissions to specific models.

In [ ]:
import boto3
import json

# Example: a least-privilege policy scoped to a single model
least_privilege_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": [
            "bedrock:InvokeModel",
            "bedrock:InvokeModelWithResponseStream"
        ],
        "Resource": [
            "arn:aws:bedrock:us-east-1::foundation-model/anthropic.claude-sonnet-4-20250514-v1:0"
        ]
    }]
}

print("Least-privilege IAM policy (scope to specific model):")
print(json.dumps(least_privilege_policy, indent=2))
print()
print("⚠️  In production, replace 'Resource: *' with the specific model ARN above.")
print("   This prevents your agent from accessing models you didn't intend.")

### 2.2 Input Validation

Always validate user input before passing it to the agent:

In [ ]:
from pydantic import BaseModel, field_validator


class AgentRequest(BaseModel):
    """Validated request model for the agent endpoint."""
    prompt: str

    @field_validator("prompt")
    @classmethod
    def validate_prompt(cls, v: str) -> str:
        # Enforce length limits to prevent token abuse
        if len(v) > 2000:
            raise ValueError("Prompt must be 2000 characters or fewer")
        if len(v.strip()) == 0:
            raise ValueError("Prompt cannot be empty")
        return v.strip()


# Usage in FastAPI
# @app.post("/weather")
# async def get_weather(request: AgentRequest):
#     response = agent(request.prompt)
#     return PlainTextResponse(content=str(response))

# Test validation
try:
    AgentRequest(prompt="")
except Exception as e:
    print(f"Validation caught empty prompt: {e}")

try:
    AgentRequest(prompt="x" * 3000)
except Exception as e:
    print(f"Validation caught long prompt: {e}")

valid = AgentRequest(prompt="What's the weather in Seattle?")
print(f"Valid prompt: '{valid.prompt}'")

### 2.3 Tool Permissions

Review what each tool can do and restrict accordingly. This example creates a domain-restricted HTTP tool:

In [ ]:
from strands import Agent, tool
from urllib.parse import urlparse
import requests as req_lib

ALLOWED_DOMAINS = ["api.weather.gov", "api.open-meteo.com"]


@tool
def safe_http_request(url: str, method: str = "GET") -> str:
    """Make HTTP requests to approved weather APIs only.

    Args:
        url: URL to request (must be from approved domains)
        method: HTTP method (GET only)
    """
    parsed = urlparse(url)
    if parsed.hostname not in ALLOWED_DOMAINS:
        return f"Error: Domain {parsed.hostname} is not in the approved list: {ALLOWED_DOMAINS}"

    if method.upper() != "GET":
        return "Error: Only GET requests are allowed."

    response = req_lib.get(url, timeout=10)
    return response.text[:5000]  # Limit response size


# Test: approved domain works
result = safe_http_request(url="https://api.weather.gov/points/47.6062,-122.3321")
print(f"✅ Approved domain: got {len(result)} chars")

# Test: unapproved domain is blocked
result = safe_http_request(url="https://evil.example.com/steal-data")
print(f"🚫 Blocked domain: {result}")

## 3. Performance Optimization

### 3.1 Streaming for Responsiveness

For web applications, streaming reduces perceived latency significantly:

In [ ]:
import asyncio
from strands import Agent


async def demonstrate_streaming():
    """Show how stream_async delivers tokens incrementally."""
    agent = Agent(
        system_prompt="You are a helpful assistant. Keep responses brief.",
    )

    print("Streaming response:")
    print("-" * 40)

    async for event in agent.stream_async("Explain why streaming matters in 2 sentences."):
        if "data" in event:
            print(event["data"], end="", flush=True)

    print("\n" + "-" * 40)
    print("Done!")


# Run the async demo
await demonstrate_streaming()

### 3.2 Conversation Management Tuning

The `SlidingWindowConversationManager` keeps your context window under control. Key parameters:

- **`window_size`** — number of message turns to keep (default: unlimited)
- Tool-pair preservation — `toolUse`/`toolResult` pairs are kept together during trimming

For tool-heavy agents (web browsing, screenshots), consider the `per_turn` parameter for proactive management.

In [ ]:
from strands import Agent
from strands.agent.conversation_manager import SlidingWindowConversationManager

# For a chatbot with moderate history needs
chatbot_manager = SlidingWindowConversationManager(window_size=20)

# For a tool-heavy agent that generates lots of intermediate messages
tool_heavy_manager = SlidingWindowConversationManager(window_size=8)

agent = Agent(
    system_prompt="You are a helpful assistant.",
    conversation_manager=chatbot_manager,
)

# Simulate multiple turns
agent("My name is Alice.")
agent("I live in Seattle.")
agent("I work as an engineer.")
response = agent("What do you know about me?")
print(response)

### 3.3 Error Handling

Production agents need robust error handling with proper logging:

In [ ]:
import logging
from strands import Agent

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)


def invoke_agent_safely(agent: Agent, prompt: str) -> str:
    """Invoke an agent with production error handling.

    Args:
        agent: Configured Strands agent
        prompt: User input

    Returns:
        Agent response or error message
    """
    try:
        result = agent(prompt)
        return str(result)

    except Exception as e:
        error_type = type(e).__name__
        logger.error(
            "Agent invocation failed",
            extra={
                "error_type": error_type,
                "error_message": str(e),
                "prompt_length": len(prompt),
            },
        )

        # Return user-friendly error (don't expose internals)
        if "throttl" in str(e).lower():
            return "The service is temporarily busy. Please try again in a moment."
        elif "token" in str(e).lower():
            return "Your request was too complex. Please try a simpler question."
        else:
            return "An error occurred processing your request. Please try again."


# Demo
agent = Agent(system_prompt="You are a helpful assistant.")
result = invoke_agent_safely(agent, "Hello!")
print(result)

## 4. Observability

In production, you need visibility into:
- **Latency** — how long agent invocations take
- **Token usage** — for cost tracking
- **Error rates** — to detect issues early
- **Tool execution** — which tools are called and how often

### 4.1 Structured Logging

In [ ]:
import time
import json
import logging
from strands import Agent

logger = logging.getLogger("agent.metrics")
logging.basicConfig(level=logging.INFO)


def invoke_with_metrics(agent: Agent, prompt: str, request_id: str) -> str:
    """Invoke agent and emit structured metrics."""
    start_time = time.time()

    try:
        result = agent(prompt)
        duration_ms = (time.time() - start_time) * 1000

        # Emit structured log for CloudWatch Insights / Datadog / etc.
        logger.info(
            json.dumps({
                "event": "agent_invocation",
                "request_id": request_id,
                "duration_ms": round(duration_ms, 2),
                "prompt_length": len(prompt),
                "response_length": len(str(result)),
                "status": "success",
            })
        )

        return str(result)

    except Exception as e:
        duration_ms = (time.time() - start_time) * 1000
        logger.error(
            json.dumps({
                "event": "agent_invocation",
                "request_id": request_id,
                "duration_ms": round(duration_ms, 2),
                "status": "error",
                "error_type": type(e).__name__,
            })
        )
        raise


# Demo
agent = Agent(system_prompt="You are a helpful assistant. Be brief.")
result = invoke_with_metrics(agent, "What is 2+2?", request_id="req-001")
print(f"Response: {result}")

### 4.2 CloudWatch Integration

For Fargate/Lambda deployments, structured JSON logs are automatically ingested by CloudWatch Logs. You can then:

1. **Create metric filters** to track `duration_ms`, error rates, etc.
2. **Set alarms** on p99 latency or error rate thresholds
3. **Use CloudWatch Insights** to query across invocations:

> **📖 Reference only** — run this query in the [CloudWatch Logs Insights console](https://console.aws.amazon.com/cloudwatch/home#logsV2:logs-insights), not in this notebook.

```
fields @timestamp, request_id, duration_ms, status
| filter event = "agent_invocation"
| stats avg(duration_ms) as avg_latency, count(*) as total by status
| sort total desc
```

For deeper tracing (model calls, tool calls, token usage), see the [Observability tutorial](../08-observability/).

## 5. Cost Optimization

### 5.1 Model Selection by Task Complexity

Not every request needs the most capable (and expensive) model:

In [ ]:
from strands import Agent
from strands.models import BedrockModel

# Cost-effective model for simple tasks
light_model = BedrockModel(
    model_id="us.amazon.nova-pro-v1:0",
    region_name="us-east-1",
    temperature=0.1,
    max_tokens=1024,
)

# Capable model for complex reasoning
heavy_model = BedrockModel(
    model_id="anthropic.claude-sonnet-4-20250514-v1:0",
    region_name="us-east-1",
    temperature=0.3,
    max_tokens=2048,
)


def route_to_model(prompt: str) -> Agent:
    """Simple complexity-based routing.

    Route simple queries to a cheaper model, complex ones to a capable model.
    """
    # Simple heuristic: short prompts with common patterns → light model
    simple_indicators = ["what is", "define", "list", "how many"]
    is_simple = (
        len(prompt.split()) < 20
        and any(prompt.lower().startswith(ind) for ind in simple_indicators)
    )

    model = light_model if is_simple else heavy_model
    return Agent(
        model=model,
        system_prompt="You are a helpful assistant. Be concise.",
    )


# Demo routing
simple_agent = route_to_model("What is the capital of France?")
print(f"Simple query → {simple_agent.model.config.get('model_id', 'default')}")

complex_agent = route_to_model(
    "Analyze the trade-offs between Lambda and Fargate for deploying "
    "a multi-turn conversational agent that needs to maintain session state "
    "across requests and handle streaming responses."
)
print(f"Complex query → {complex_agent.model.config.get('model_id', 'default')}")

# Actually invoke to prove it works
print("\n--- Invoking simple agent ---")
print(simple_agent("What is the capital of France?"))

### 5.2 Token Management Tips

> **📖 Reference only** — these are guidelines to apply when configuring your agent. Not executable code.

| Technique | Impact | How |
|-----------|--------|-----|
| Shorter system prompts | Reduces input tokens per request | Remove verbose instructions; be directive |
| `window_size` tuning | Limits accumulated context | Set based on actual conversation depth needed |
| Tool docstring brevity | Fewer tokens per tool spec | Keep tool descriptions concise but clear |
| Response length limits | Controls output tokens | Use `max_tokens` and prompt instructions |
| Prompt caching | Reduces repeated token costs | Use `SystemContentBlock` with `cachePoint` for prompts > 1024 tokens |

## 6. Production Checklist

> **📖 Reference only** — this is a review checklist to go through before deploying to production. Not executable code.

Before going live, verify:

### Configuration
- [ ] Model ID explicitly set (not relying on defaults)
- [ ] Temperature tuned for your use case
- [ ] `max_tokens` set high enough for tool calls + response
- [ ] Conversation manager configured with appropriate `window_size`
- [ ] Tools explicitly listed (no auto-discovery)

### Security
- [ ] IAM role scoped to specific model ARNs
- [ ] Input validation on all user-facing endpoints
- [ ] Tool permissions follow least-privilege
- [ ] No secrets in code or environment variables (use Secrets Manager)
- [ ] Container runs as non-root user

### Performance
- [ ] Streaming enabled for interactive use cases
- [ ] Health check endpoint configured
- [ ] Error handling returns user-friendly messages
- [ ] Timeout configured appropriately for compute service

### Observability
- [ ] Structured logging with request IDs
- [ ] Latency and error rate metrics
- [ ] CloudWatch alarms on key thresholds
- [ ] Token usage tracking for cost monitoring

### Reliability
- [ ] Multiple instances/tasks for high availability (Fargate)
- [ ] Circuit breaker with rollback enabled
- [ ] Graceful degradation on model throttling

## Summary

In this notebook, you've learned production hardening patterns:

| Area | Key Takeaway |
|------|-------------|
| **Configuration** | Be explicit about model, temperature, tokens, and conversation management |
| **Security** | Least-privilege IAM, input validation, tool restrictions |
| **Performance** | Streaming for UX, conversation windowing for stability |
| **Observability** | Structured JSON logs, latency metrics, CloudWatch integration |
| **Cost** | Route by complexity, tune token budgets, cache prompts |

These patterns apply regardless of which deployment target (Lambda or Fargate) you chose in Notebook 02.

---

**Congratulations!** You've completed the Production Deployment Patterns tutorial. You now know how to take a Strands agent from local development to a hardened production deployment on AWS.